# Aralin 16 - Pag-deploy ng Mga Scalable na Ahente kasama ang Microsoft Foundry

Sa notebook na ito, gagawa ka ng isang **production-ready customer support agent** para sa kathang-isip na kumpanya na **Contoso**. Hindi tulad ng mga naunang aralin, ang punto dito ay hindi ang reasoning loop ng ahente—ito ay ang lahat ng nakabalot *sa paligid* nito na nagpapagawa ng isang ahente na ligtas patakbuhin sa malakihang sukat:

1. **Pagtawag ng tool** — order lookups at paggawa ng ticket.
2. **RAG** — mga sagot mula sa isang knowledge base.
3. **Memorya** — pag-alala sa customer sa bawat turn.
4. **Pag-ruta ng modelo** — ipadala ang mga simpleng kahilingan sa isang maliit na modelo, at ang mga kumplikado sa isang malaking modelo.
5. **Response caching** — pagsagot sa mga paulit-ulit na tanong nang hindi tumatawag sa modelo.
6. **Pag-apruba ng tao** — ang mga refund na lampas sa threshold ay hihinto muna para sa pag-apruba.
7. **Evaluation gate** — isang offline test set na humaharang sa isang masamang release.
8. **Observability** — OpenTelemetry tracing sa paligid ng bawat kahilingan.

Bawat seksyon ay nakahiwalay at pwedeng patakbuhin nang sarili. Basahin ang bawat linya—ang mga production primitives ay sinadya na gawing maliit.


## Setup

Bago patakbuhin ang notebook na ito, siguraduhin na mayroon kang:

1. **Isang Microsoft Foundry project** na may naka-deploy na chat model (hal. `gpt-5-mini`).
2. **Nakalog-in gamit ang Azure CLI** — patakbuhin ang `az login` sa iyong terminal.
3. **Na-set ang mga kailangang environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ang endpoint ng iyong Microsoft Foundry project.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong naka-deploy na modelo.

Ginagamit ng seksyong RAG ang **Azure AI Search** kapag ang `AZURE_SEARCH_SERVICE_ENDPOINT` at `AZURE_SEARCH_API_KEY` ay na-set, at bumabawi sa in-memory search upang tumakbo ang notebook nang walang Search resource.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Mga Kasangkapan

Ang mga kasangkapang pang-produksyon ay gumagawa ng totoong gawain laban sa mga totoong sistema. Dito ay ginagaya natin ang isang order database at isang ticketing system gamit lang ang mga plain na Python functions. Ang `@tool` decorator ay nagpapakita sa mga ito sa ahente.

Pansinin na ang `issue_refund` ay gumagamit ng `approval_mode="always_require"` para sa mga refund na lampas sa isang threshold — ito ang human-in-the-loop na primitive na ipapagamit natin pagkatapos.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Patakaran sa Base ng Kaalaman

Ang mga tanong tungkol sa patakaran ("ano ang iyong window ng pagbalik?") ay dapat sagutin mula sa isang awtoritatibong pinagmulan, hindi mula sa memorya ng modelo. Nagbabalot kami ng isang maliit na base ng kaalaman bilang isang kasangkapang panghanap.

Sa produksyon ito ay **Azure AI Search**; dito nagbibigay kami ng in-memory keyword search upang tumakbo ang notebook kahit saan, at awtomatikong maglilipat sa Azure AI Search kapag naroroon ang mga environment variables.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memorya

Ang isang support agent na nakakalimot kung sino ang kausap ay isang masamang support agent. Nagtatago kami ng maliit na imbakan ng profile para sa bawat customer at ini-inject ang isang maikling buod sa mga tagubilin ng agent. Sa produksyon ito ay isang memory service (tingnan ang Lesson 13); dito isang dict ang nagpapakita ng pattern.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Pag-ruta ng Modelo at Pag-cache ng Tugon

Dalawang lever ng gastos na nakakabit sa isang tagapamahala ng kahilingan:

- **Pag-ruta**: isang murang heuristic classifier ang nagdedesisyon kung ang isang kahilingan ay nangangailangan ng maliit o malaking modelo.
- **Pag-cache**: ang mga normalisadong paulit-ulit na mga tanong ay sinisilbihan diretso mula sa cache nang walang tawag sa modelo.

Ang classifier dito ay sadyang simple. Sa produksyon susuriin mo ito laban sa trapiko at maaaring palitan ito ng Model Router ng Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Ang Ahente, Pag-apruba ng Tao, at Obserbabilidad

Ngayon ay pinagsasama natin ang ahente mula sa mga tool sa itaas at binabalutan ang bawat kahilingan ng isang OpenTelemetry span. Ang function na `handle_support_request` ang tagapangasiwa ng kahilingan sa produksyon: cache → ruta → trace → patakbuhin → cache.

Ang pag-apruba ng tao ay hinahandle ng framework: dahil ang `issue_refund` ay may `approval_mode="always_require"`, ang pagtakbo ay humihinto at nagpapakita ng isang kahilingan ng pag-apruba na ating nilulutas nang hayagan.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Evaluation Gate

Ito ang release gate mula sa leksyon: isang offline na test set ang sumusuri sa ahente, at ang deployment ay magpapatuloy lamang kung ang pass rate ay makalampas sa isang threshold. Ang scorer dito ay isang simpleng keyword-overlap check upang mapanatiling self-contained ang notebook; sa produksyon gagamit ka ng LLM-as-judge o isang framework evaluator (tingnan ang Leksiyon 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Pagsasama-sama: Isang Nalinlang na Pagpapalabas

Ipinapakita ng cell sa ibaba ang buong loop na inilalarawan ng aralin: patakbuhin ang evaluation gate, at i-"deploy" lamang kung ito ay pumasa. Ito ang pattern na patatakbuhin mo sa CI bago i-promote ang isang bersyon ng agent sa Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Buod

Nalikha mo ang isang handang gumamit na customer support agent na may lahat ng operasyonal na alalahanin na naka-wire:

- **Mga Tools, RAG, at memorya** ang nagbibigay kakayahan at konteksto sa ahente.
- **Pag-route ng modelo at caching** ang nagpapanatili ng mababang pagkaantala at gastos.
- **Pag-apruba ng tao** ang nagbabantay sa mga aksyong may malaking panganib tulad ng malalaking refund.
- **Ang evaluation gate** ang humaharang sa mga maling release bago ito ipadala.
- **Pagsubaybay** ang nagpapakita ng bawat kahilingan.

### Hamon

Palawakin ang ahenteng ito upang:

1. **Suportahan ang maraming modelo** — magdagdag ng ikatlong "reasoning" na antas at i-route ang mga eskalasiyon/reklamo dito.
2. **Magdagdag ng evaluation gates** — palawakin ang `TEST_CASES` para isama ang mga senaryo ng pag-apruba ng refund at tiyaking nahuhuli ng gate ang mga regression.
3. **Magdagdag ng cost-aware routing** — subaybayan ang tantyang gastos kada kahilingan (maliit kumpara sa malaki at cache) at mag-print ng ulat ng gastos matapos ang isang batch ng halo-halong mga query.

Sa susunod na aralin, dadalhin ka sa kabaligtarang paglalakbay at patatakbuhin ang isang ahente nang ganap sa iyong sariling makina gamit ang Microsoft Foundry Local at Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
